# Data Quality Check

In [27]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

In [28]:
BASE = Path("/home/ftp_246/data_5/多模態2026_ig_top200")

METADATA_DIR = BASE / "Metadata_by_Influencer"
VIDEO_DIR = BASE / "Video_data_fullest/ig_vedios"
JSON_DIR = BASE / "Prompt_Description/Fullset_v20"

PROFILE_PATH = Path(
    "/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv"
)

# 檢查路徑
paths = {
    "Profile": PROFILE_PATH,
    "Metadata": METADATA_DIR,
    "MP4": VIDEO_DIR,
    "JSON": JSON_DIR
}

for name, path in paths.items():
    print(f"{name}: {path.exists()} | {path}")

if not all(path.exists() for path in paths.values()):
    raise FileNotFoundError("請先修正不存在的資料路徑")

# 載入網紅 Profile，保留所有原始欄位
df_profile = pd.read_csv(
    PROFILE_PATH,
    dtype={"ig_id": "string", "ig_strong_id": "string"}
)

df_profile["ig_id"] = df_profile["ig_id"].str.strip()

print("Profile rows:", len(df_profile))

Profile: True | /home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv
Metadata: True | /home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer
MP4: True | /home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios
JSON: True | /home/ftp_246/data_5/多模態2026_ig_top200/Prompt_Description/Fullset_v20
Profile rows: 203


## 讀取所有 Metadata，產生預期影片檔名

In [29]:
def format_modified_time(value):
    """將 modified_time_tw 轉為 YYYYMMDDHHMMSS。"""
    if pd.isna(value):
        return None

    s = str(value).strip()

    # 已經是 14 位數字
    if re.fullmatch(r"\d{14}", s):
        return s

    # 一般日期時間格式
    dt = pd.to_datetime(s, errors="coerce")

    if pd.notna(dt):
        return dt.strftime("%Y%m%d%H%M%S")

    return None


metadata_frames = []
metadata_errors = []

for csv_path in METADATA_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(
            csv_path,
            dtype={
                "id": "string",
                "media_id": "string",
                "post_owner.username": "string"
            },
            low_memory=False
        )

        required = [
            "id",
            "media_id",
            "post_owner.username",
            "modified_time_tw"
        ]

        missing = [c for c in required if c not in df.columns]

        if missing:
            metadata_errors.append({
                "file": str(csv_path),
                "error": f"Missing columns: {missing}"
            })
            continue

        df["metadata_path"] = str(csv_path)
        metadata_frames.append(df)

    except Exception as e:
        metadata_errors.append({
            "file": str(csv_path),
            "error": str(e)
        })

if not metadata_frames:
    raise ValueError("沒有成功讀取任何 Metadata CSV")

df_metadata = pd.concat(
    metadata_frames, ignore_index=True
)

# 統一欄位名稱
df_metadata = df_metadata.rename(columns={
    "id": "post_id",
    "post_owner.username": "ig_id"
})

for col in ["ig_id", "post_id", "media_id"]:
    df_metadata[col] = df_metadata[col].astype("string").str.strip()

df_metadata["modified_timestamp"] = (
    df_metadata["modified_time_tw"]
    .apply(format_modified_time)
)

# 確保組合檔名所需的欄位完整
valid_name = (
    df_metadata["ig_id"].notna()
    & df_metadata["post_id"].notna()
    & df_metadata["media_id"].notna()
    & df_metadata["modified_timestamp"].notna()
)

df_metadata["expected_video_filename"] = pd.NA

df_metadata.loc[valid_name, "expected_video_filename"] = (
    df_metadata.loc[valid_name, "ig_id"]
    + "-"
    + df_metadata.loc[valid_name, "modified_timestamp"]
    + "-"
    + df_metadata.loc[valid_name, "media_id"]
    + ".mp4"
)

print("Metadata rows:", len(df_metadata))
print("CSV read errors:", len(metadata_errors))
print("Rows with valid filenames:", valid_name.sum())

display(
    df_metadata[
        [
            "ig_id",
            "post_id",
            "media_id",
            "modified_time_tw",
            "expected_video_filename"
        ]
    ].head()
)

Metadata rows: 19443
CSV read errors: 0
Rows with valid filenames: 19443


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename
0,traveggo,916191564143111,916191564143111,2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4
1,traveggo,3243955289117081,3243955289117081,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4
2,traveggo,1604802410544990,1604802410544990,2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4
3,traveggo,849685401437704,849685401437704,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4
4,traveggo,1452518233264809,1452518233264809,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4


## 掃描 MP4 與 JSON，建立 Post-level Inventory

In [30]:
# 1. 掃描所有 MP4
mp4_rows = []

for path in VIDEO_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".mp4":
        mp4_rows.append({
            "ig_id": path.parent.name,
            "video_filename": path.name,
            "video_path": str(path),
            "video_size_mb": round(
                path.stat().st_size / (1024 ** 2), 2
            )
        })

df_mp4 = pd.DataFrame(
    mp4_rows,
    columns=[
        "ig_id", "video_filename",
        "video_path", "video_size_mb"
    ]
)

# 2. 掃描 JSON
json_rows = []
json_errors = []

for path in JSON_DIR.rglob("*.json"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        video_name = data.get("videoName")

        if not isinstance(video_name, str):
            video_name = None

        json_rows.append({
            "ig_id": path.parent.name,
            "video_filename": (
                Path(video_name).name if video_name else None
            ),
            "json_path": str(path)
        })

    except (OSError, ValueError, TypeError) as e:
        json_errors.append({
            "file": str(path),
            "error": str(e)
        })

df_json = pd.DataFrame(
    json_rows,
    columns=["ig_id", "video_filename", "json_path"]
)

# 3. 以 ig_id + 完整檔名作為匹配鍵
keys = ["ig_id", "video_filename"]

# 找出重複鍵，避免 merge 造成多對多膨脹
mp4_key_counts = df_mp4.groupby(keys, dropna=False).size()
json_key_counts = df_json.groupby(keys, dropna=False).size()

mp4_unique = df_mp4[
    ~df_mp4.set_index(keys).index.isin(
        mp4_key_counts[mp4_key_counts > 1].index
    )
]

json_unique = df_json[
    ~df_json.set_index(keys).index.isin(
        json_key_counts[json_key_counts > 1].index
    )
]

# 4. 建立貼文清單
df_post_inventory = df_metadata.copy()

df_post_inventory["video_filename"] = (
    df_post_inventory["expected_video_filename"]
)

df_post_inventory = df_post_inventory.merge(
    mp4_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

df_post_inventory = df_post_inventory.merge(
    json_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

# 5. 建立完整度旗標
df_post_inventory["has_mp4"] = (
    df_post_inventory["video_path"].notna()
)

df_post_inventory["has_json"] = (
    df_post_inventory["json_path"].notna()
)

df_post_inventory["is_complete"] = (
    df_post_inventory["has_mp4"]
    & df_post_inventory["has_json"]
)

df_post_inventory["match_status"] = np.select(
    [
        df_post_inventory["is_complete"],
        df_post_inventory["has_mp4"],
        df_post_inventory["has_json"]
    ],
    [
        "complete",
        "mp4_only",
        "json_only"
    ],
    default="missing_both"
)

# 6. 重複貼文標記
df_post_inventory["duplicate_post_id"] = (
    df_post_inventory.duplicated(
        ["ig_id", "post_id"], keep=False
    )
)

# 簡化輸出欄位，保留研究所需資訊
post_columns = [
    "ig_id",
    "post_id",
    "media_id",
    "short_code",
    "text",
    "duration",
    "tags",
    "modified_time_tw",
    "video_filename",
    "video_path",
    "json_path",
    "video_size_mb",
    "has_mp4",
    "has_json",
    "is_complete",
    "match_status",
    "duplicate_post_id",
    "metadata_path"
]

df_post_inventory = df_post_inventory[
    [c for c in post_columns if c in df_post_inventory.columns]
]

print("Metadata rows:", len(df_post_inventory))
print("MP4 files:", len(df_mp4))
print("JSON records:", len(df_json))
print("Complete rows:", df_post_inventory["is_complete"].sum())

display(df_post_inventory.head())

Metadata rows: 19443
MP4 files: 19312
JSON records: 19116
Complete rows: 19102


,ig_id,post_id,media_id,short_code,text,duration,tags,modified_time_tw,video_filename,video_path,json_path,video_size_mb,has_mp4,has_json,is_complete,match_status,duplicate_post_id,metadata_path
0,traveggo,916191564143111,916191564143111,DE2RQZzy7ik,#同居觀察日記 Ep1\n同居是感情必修學分\n兩人住進同一個屋簷下\n首先迎來的 絕對是習...,90.02,{'0': '1833997590677795'},2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,17.34,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
1,traveggo,3243955289117081,3243955289117081,DFK56DuvHOw,#同居觀察日記 Ep2\n經歷女生月經週期像換了4個女友……\n\n#情侶日常 #vegan...,31.81,NaN,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,4.79,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
2,traveggo,1604802410544990,1604802410544990,DFNcyBZSSvJ,#同居觀察日記 Ep3\n我以為我在女生界洗澡算快了\n同居才發現男女洗澡速度真的差好多😂\...,89.20,{'0': '459340223188257'},2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,16.42,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
3,traveggo,849685401437704,849685401437704,DFXxOBDv1YN,帥哥帥妹祝大家新年快樂❤️\n2025 蛇麼都順利 平安 健康 心中有愛,15.16,NaN,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,2.81,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
4,traveggo,1452518233264809,1452518233264809,DGYHxkTvIr9,#同居觀察日記 Ep4 \n啊不是要保護我？？？？？？,40.87,NaN,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,5.06,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...


## 產生網紅完整統計表

In [31]:
# 1. 以唯一 post_id 計算貼文數
post_stats = (
    df_post_inventory
    .groupby("ig_id", dropna=False)
    .agg(
        metadata_row_count=("post_id", "size"),
        metadata_post_count=("post_id", "nunique"),
        matched_video_count=("has_mp4", "sum"),
        matched_json_count=("has_json", "sum"),
        fully_matched_count=("is_complete", "sum"),
        duplicate_post_rows=("duplicate_post_id", "sum")
    )
    .reset_index()
)

# 2. MP4 實際檔案數
mp4_stats = (
    df_mp4.groupby("ig_id")
    .agg(
        mp4_count=("video_path", "size"),
        total_video_size_mb=("video_size_mb", "sum")
    )
    .reset_index()
)

# 3. JSON 實際檔案數
json_stats = (
    df_json.groupby("ig_id")
    .agg(json_count=("json_path", "size"))
    .reset_index()
)

# 4. 合併 Profile 與所有統計
# 使用 outer，避免遺漏 Profile 以外的帳號
df_influencer_summary = (
    df_profile
    .merge(
        post_stats,
        on="ig_id",
        how="outer",
        indicator="profile_merge_status"
    )
    .merge(mp4_stats, on="ig_id", how="outer")
    .merge(json_stats, on="ig_id", how="outer")
)

count_columns = [
    "metadata_row_count",
    "metadata_post_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "duplicate_post_rows",
    "mp4_count",
    "json_count"
]

for col in count_columns:
    df_influencer_summary[col] = (
        df_influencer_summary[col]
        .fillna(0)
        .astype(int)
    )

df_influencer_summary["total_video_size_mb"] = (
    df_influencer_summary["total_video_size_mb"]
    .fillna(0)
)

# 5. 計算完整率
denominator = (
    df_influencer_summary["metadata_post_count"]
    .replace(0, np.nan)
)

df_influencer_summary["complete_rate"] = (
    df_influencer_summary["fully_matched_count"]
    / denominator
)

# 6. 標記是否有可用資料
df_influencer_summary["has_usable_video"] = (
    df_influencer_summary["fully_matched_count"] > 0
)

# 7. 將重要欄位放在前面，其餘 Profile 欄位保留
priority_cols = [
    "ig_id",
    "ig_strong_id",
    "AsiaKOL_category",
    "Followers",
    "Following",
    "posts",
    "metadata_post_count",
    "mp4_count",
    "json_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "complete_rate",
    "has_usable_video",
    "total_video_size_mb"
]

priority_cols = [
    c for c in priority_cols
    if c in df_influencer_summary.columns
]

other_cols = [
    c for c in df_influencer_summary.columns
    if c not in priority_cols
]

df_influencer_summary = (
    df_influencer_summary[priority_cols + other_cols]
    .sort_values(
        "fully_matched_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(df_influencer_summary)

print("Total influencers:", len(df_influencer_summary))
print(
    "Influencers with complete videos:",
    df_influencer_summary["has_usable_video"].sum()
)
print(
    "Total complete post rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

,ig_id,ig_strong_id,AsiaKOL_category,Followers,Following,posts,metadata_post_count,mp4_count,json_count,matched_video_count,...,has_usable_video,total_video_size_mb,Aisa_Order,person_name,ig_name,ig_url,AsiaKOL_followers,metadata_row_count,duplicate_post_rows,profile_merge_status
0,dr.acne_scar_queen,4833663962,"知識教育,醫療健康",189848,259,2943,639,639,639,639,...,True,4142.59,137,痘疤醫生---莊盈彥醫師,痘疤女王 莊盈彥,https://www.instagram.com/dr.acne_scar_queen/,"922,505",639,0,both
1,mayukichou,18346291,"時尚潮流,高階經理人",309258,1748,8861,538,538,538,538,...,True,2242.52,163,周小葳,葳老闆｜周品均,https://www.instagram.com/mayukichou/,"636,488",538,0,both
2,cocowine0205,2307480066,"美食料理,趣味搞笑",157798,782,1593,391,391,391,391,...,True,5889.88,98,可可酒精,可可酒精,https://www.instagram.com/cocowine0205/,"853,725",391,0,both
3,yga0721,45717781,趣味搞笑,1495461,501,8178,344,344,342,344,...,True,4807.44,1,蔡阿嘎,蔡阿嘎,https://www.instagram.com/yga0721/,"5,893,570",344,0,both
4,ladyflavor_cui,3656815405,NaN,369611,122,1378,342,342,342,342,...,True,4844.68,14,好味小姐,好味小姐脆脆,https://www.instagram.com/ladyflavor_cui/,NaN,342,0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,alvin701,1972535324,知識教育,187869,142,1329,1,0,0,0,...,False,0.00,115,艾爾文,艾爾文,https://www.instagram.com/alvin701/,"1,563,768",1,0,both
199,lynnwu0219_2,47642521277,帶貨分潤,76276,93,21,0,0,0,0,...,False,0.00,39,泱泱LYNN,泱泱LYNN,https://www.instagram.com/lynnwu0219_2/,"1,965,591",0,0,left_only
200,skauramomo,515352630,"時尚潮流,表演藝術",0,0,0,35,0,0,0,...,False,0.00,195,紫玥桃,紫玥桃 Cosplay,https://www.instagram.com/skauramomo/,"676,777",35,0,both
201,oldwangwang,3216171228,"趣味搞笑,時事討論",121681,57,289,1,1,1,0,...,False,5.13,177,老王-oldwang,老王 oldwong,https://www.instagram.com/oldwangwang/,"2,501,639",1,0,both


Total influencers: 203
Influencers with complete videos: 196
Total complete post rows: 19102


## 建立集中式 Data Quality Summary

In [32]:
# 重複貼文 ID
duplicate_posts = (
    df_post_inventory["duplicate_post_id"].sum()
)

# 沒有對應到 Metadata 的 MP4
expected_keys = df_metadata[
    ["ig_id", "expected_video_filename"]
].dropna().drop_duplicates().rename(
    columns={"expected_video_filename": "video_filename"}
)

orphan_mp4 = df_mp4.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_mp4_count = (
    orphan_mp4["_merge"] == "left_only"
).sum()

# 沒有對應到 Metadata 的 JSON
orphan_json = df_json.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_json_count = (
    orphan_json["_merge"] == "left_only"
).sum()

# Profile 沒有出現的網紅
profile_ids = set(df_profile["ig_id"].dropna())

all_data_ids = (
    set(df_metadata["ig_id"].dropna())
    | set(df_mp4["ig_id"].dropna())
    | set(df_json["ig_id"].dropna())
)

unknown_influencers = all_data_ids - profile_ids

# Profile 中重複的識別碼
duplicate_strong_ids = (
    df_profile["ig_strong_id"]
    .dropna()
    .duplicated()
    .sum()
)

quality_checks = [
    ("Metadata 總列數", len(df_metadata)),
    ("唯一網紅數（所有來源）", len(profile_ids | all_data_ids)),
    ("實際 MP4 數", len(df_mp4)),
    ("成功解析 JSON 數", len(df_json)),
    ("完整匹配貼文列數",
     int(df_post_inventory["is_complete"].sum())),
    ("缺少 MP4 的貼文列數",
     int((~df_post_inventory["has_mp4"]).sum())),
    ("缺少 JSON 的貼文列數",
     int((~df_post_inventory["has_json"]).sum())),
    ("缺少 MP4 與 JSON 的貼文列數",
     int((df_post_inventory["match_status"] == "missing_both").sum())),
    ("重複貼文 ID 列數", int(duplicate_posts)),
    ("重複 ig_strong_id 數", int(duplicate_strong_ids)),
    ("沒有 Metadata 對應的 MP4 數", int(orphan_mp4_count)),
    ("沒有 Metadata 對應的 JSON 數", int(orphan_json_count)),
    ("Profile 以外的網紅數", len(unknown_influencers)),
    ("Metadata CSV 讀取錯誤數", len(metadata_errors)),
    ("JSON 解析錯誤數", len(json_errors)),
    ("重複 MP4 匹配鍵數",
     int((mp4_key_counts > 1).sum())),
    ("重複 JSON 匹配鍵數",
     int((json_key_counts > 1).sum()))
]

df_data_quality = pd.DataFrame(
    quality_checks,
    columns=["check_item", "count"]
)

display(df_data_quality)

# 找出 Metadata 有紀錄，但找不到對應 MP4 的貼文
df_missing_videos = df_post_inventory.loc[
    ~df_post_inventory["has_mp4"]
].copy()

# 重新產生預期檔名，方便查找
df_missing_videos["expected_video_filename"] = (
    df_missing_videos["video_filename"]
)

# 選擇要顯示的欄位
columns = [
    "ig_id",
    "post_id",
    "media_id",
    "modified_time_tw",
    "expected_video_filename",
    "has_json",
    "match_status"
]

df_missing_videos = df_missing_videos[columns]

print(f"Missing MP4 posts: {len(df_missing_videos)}")

# 顯示前 20 筆缺少 MP4 的貼文
display(df_missing_videos.head(20))

,check_item,count
0,Metadata 總列數,19443
1,唯一網紅數（所有來源）,203
2,實際 MP4 數,19312
3,成功解析 JSON 數,19116
4,完整匹配貼文列數,19102
5,缺少 MP4 的貼文列數,133
6,缺少 JSON 的貼文列數,341
7,缺少 MP4 與 JSON 的貼文列數,133
8,重複貼文 ID 列數,0
9,重複 ig_strong_id 數,1


Missing MP4 posts: 133


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename,has_json,match_status
65,traveggo,4648229675409463,4648229675409463,2025-09-06 22:44:10+08:00,traveggo-20250906224410-4648229675409463.mp4,False,missing_both
302,tzu_888,1215205530776275,1215205530776275,2025-12-09 15:31:21+08:00,tzu_888-20251209153121-1215205530776275.mp4,False,missing_both
1077,wia627,1852338622114597,1852338622114597,2025-09-20 04:58:47+08:00,wia627-20250920045847-1852338622114597.mp4,False,missing_both
1989,zxsdexz,1213717130801192,1213717130801192,2026-01-05 23:09:26+08:00,zxsdexz-20260105230926-1213717130801192.mp4,False,missing_both
1991,0_shufen,1165134668817062,1165134668817062,2025-11-23 04:00:36+08:00,0_shufen-20251123040036-1165134668817062.mp4,False,missing_both
2119,1_shiuan_0,1639657417044170,1639657417044170,2025-08-19 22:25:28+08:00,1_shiuan_0-20250819222528-1639657417044170.mp4,False,missing_both
2352,2uncle987,888601933921971,888601933921971,2025-11-22 01:00:34+08:00,2uncle987-20251122010034-888601933921971.mp4,False,missing_both
3289,1105ya,1051447247181222,1051447247181222,2025-12-12 13:43:16+08:00,1105ya-20251212134316-1051447247181222.mp4,False,missing_both
3501,ackj0301_ifbbpro,1646270259713007,1646270259713007,2026-01-16 03:33:22+08:00,ackj0301_ifbbpro-20260116033322-16462702597130...,False,missing_both
3598,ala780705,1258671062799202,1258671062799202,2025-01-01 14:33:38+08:00,ala780705-20250101143338-1258671062799202.mp4,False,missing_both


## 最終統計表

In [33]:
print("=== SPoD Dataset Overview ===")

print(
    "Total influencers:",
    df_influencer_summary["ig_id"].nunique()
)

print(
    "Influencers with complete data:",
    df_influencer_summary["has_usable_video"].sum()
)

print(
    "Total metadata posts:",
    df_influencer_summary["metadata_post_count"].sum()
)

print(
    "Total MP4 files:",
    df_influencer_summary["mp4_count"].sum()
)

print(
    "Total JSON files:",
    df_influencer_summary["json_count"].sum()
)

print(
    "Total fully matched rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

print(
    "Total video storage (GB):",
    round(
        df_influencer_summary["total_video_size_mb"].sum() / 1024,
        2
    )
)

=== SPoD Dataset Overview ===
Total influencers: 203
Influencers with complete data: 196
Total metadata posts: 19443
Total MP4 files: 19312
Total JSON files: 19116
Total fully matched rows: 19102
Total video storage (GB): 235.73
